# Predicting Pokémon battles after turn 30

Amirhosein · FDS Challenge 2026

## Summary

**Combined vote** led at **68.28%** mean grouped cross-validation accuracy on 7,996 development battles. It gained **0.40 percentage points** over 300 compact vote and made **32 more correct predictions**. The reserved holdout stayed unscored in this run.

Open with the project Python kernel and run from this checkout. Install `requirements.txt`
in that environment and put the official files in `data/raw/` or set `POKEMON_DATA_DIR`.
The default run writes an audit under `outputs/amirhosein/`. Holdout scoring and submission
generation are opt-in at the end.

## Context and methods

Predict whether player 1 wins from the supplied P1 team, P2 lead, and observed turns 1–30.
Battle IDs and outcomes stay out of predictors. Unseen bench slots can be presumed healthy
by roster proxies, and last-seen HP may be stale. Damage and speed estimates approximate
battle mechanics. When a provider lacks metadata, its estimate can remain missing.

This consolidates the compact 300-feature study, Aimilia's feature-bundle comparison,
and the prior o3-reviewed numeric union. The compact branch excludes four opponent-roster
stat summaries backed by a legacy lookup assembled from test rosters and external data.
The Aimilia branch learns species metadata from each fitting partition and can use
metadata supplied in the battle being scored. The official type chart is the only reference input.

In [ ]:
from datetime import datetime
from pathlib import Path
from hashlib import file_digest, sha256 as hash_bytes
from importlib.metadata import version
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display, Markdown
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from threadpoolctl import threadpool_limits

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/pokemon_challenge').is_dir() and (p / 'configs/validation.json').is_file()), None)
if root is None:
    raise FileNotFoundError('Start the notebook from inside the FDS Challenge checkout.')
support = root / 'notebooks/_amir_support'
sys.path[:0] = [str(root / 'src'), str(support)]

from pokemon_challenge.data import iter_battles
from pokemon_challenge.models.classifiers import make_pipeline
from pokemon_challenge.paths import data_dir, output_dir
from pokemon_challenge.validation.splits import make_splits
from emily_enhanced import EmilyFeatures
from feature_catalog import feature_dictionary
from feature_selection import CorrelationUtilitySelector
from modeling import BattleFeatureLibrary, model_candidates
from study import observation_digest, observed_battle

settings = json.loads((root / 'configs/validation.json').read_text())
seed, n_folds = settings['seed'], settings['n_splits']
data = data_dir()
chart = data / 'type_chart_gen2.csv'
run = output_dir('amirhosein', datetime.now().strftime('%Y%m%d_%H%M%S_%f_final'))
kernel = get_ipython()
audit_history_start = max(1, len(kernel.history_manager.input_hist_raw) - 1) if kernel else None
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white', 'figure.dpi': 110})
blue, gold, ink = '#235789', '#99620B', '#273746'

## Data and validation

Keep the team’s original stratified holdout. Quarantine any development observation
that matches it, then use five stratified group folds so identical permitted
observations cannot cross a training/validation boundary. The digest ignores P1
roster order. These development folds have been used for prior experiments, and
the team has previously scored the shared holdout. Treat the model comparison
as exploratory.

The split and prediction exports both number development folds from 1 to 5.

In [ ]:
raw = list(iter_battles(data / 'train.jsonl'))
ids = np.array([battle['battle_id'] for battle in raw])
y = np.array([battle['player_won'] for battle in raw], dtype=int)
assert pd.Index(ids).is_unique and set(y) == {0, 1}

hashes = np.array([observation_digest(battle) for battle in raw])
_, holdout, _, _ = make_splits(ids, y, seed=seed, n_splits=n_folds, holdout_size=settings['holdout_size'])
dev = np.flatnonzero(~np.isin(hashes, hashes[holdout]))
battles = pd.DataFrame({'battle': raw})
dev_battles, y_dev = battles.iloc[dev], y[dev]
groups = hashes[dev]
folds = list(StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
             .split(dev_battles, y_dev, groups))

split = pd.DataFrame({'battle_id': ids, 'partition': 'holdout_duplicate_quarantine',
                      'validation_fold': -1, 'observation_sha256': hashes})
split.loc[holdout, 'partition'] = 'holdout'
split.loc[dev, 'partition'] = 'development'
for fold, (fit_rows, valid_rows) in enumerate(folds, 1):
    assert not set(groups[fit_rows]).intersection(groups[valid_rows])
    split.loc[dev[valid_rows], 'validation_fold'] = fold
assert (split.loc[dev, 'validation_fold'] > 0).all()
split.to_csv(run / 'splits.csv', index=False)
display(split['partition'].value_counts().rename(index={
    'development': 'Development', 'holdout': 'Reserved holdout',
    'holdout_duplicate_quarantine': 'Quarantined duplicates',
}).to_frame('Battles'))
print(f'Development P1 win rate: {y_dev.mean():.2%}')

## Model recipes

The 300-feature branch combines logistic regression, histogram gradient boosting
and XGBoost with equal weights. The 106-feature baseline and 276-feature Aimilia
bundle each average logistic and histogram boosting probabilities. The numeric
union selects 320 features across both libraries and fits logistic regression.
The union vote averages the 276-feature vote and union logistic model.
The combined vote gives half its weight to the 300-feature vote and half to the
union vote. A probability above 0.50 predicts a P1 win.

Both selectors remove empty, constant and duplicate training vectors, then
rank utility with logistic coefficients, ExtraTrees importance and univariate
F scores. They cap absolute pairwise Pearson correlation at 0.90 on the
training-median-imputed matrix. Selection, species lookup, imputation and
scaling are refitted inside each fold.

In [ ]:
model_300 = {f'300 {item.name}': (item.model, item.params)
             for item in model_candidates() if item.model != 'soft_voting'}
aimilia_lr = ('logistic_regression', {'C': .01, 'max_iter': 4000})
aimilia_hgb = ('hist_gradient_boosting', {
    'max_iter': 500, 'learning_rate': .04, 'max_leaf_nodes': 3, 'min_samples_leaf': 150,
    'l2_regularization': 20., 'early_stopping': False,
})
aimilia_sizes = {'106': 106, '276': 276}
aimilia_models = {f'{bundle} {name}': recipe for bundle in aimilia_sizes
                  for name, recipe in [('logistic', aimilia_lr), ('histogram', aimilia_hgb)]}
union_recipe = ('logistic_regression', {'C': .02, 'max_iter': 3000})
members = {**model_300, **aimilia_models, '320 union logistic': union_recipe}
recipes = {
    '106 baseline vote': {'106 logistic': .5, '106 histogram': .5},
    '276 Aimilia vote': {'276 logistic': .5, '276 histogram': .5},
    '300 compact vote': dict.fromkeys(model_300, 1 / 3),
    'Union vote': {'276 logistic': .25, '276 histogram': .25, '320 union logistic': .5},
}
recipes['Combined vote'] = {
    **dict.fromkeys(model_300, 1 / 6),
    **{name: weight / 2 for name, weight in recipes['Union vote'].items()},
}
candidates = [*recipes, *model_300, '320 union logistic']
library = BattleFeatureLibrary(chart, include_p2_roster_stats=False)

def amir_matrix(frame):
    return pd.DataFrame([library(observed_battle(battle)) for battle in frame['battle']],
                        index=frame.index, dtype=float)

def select_pair(train, valid, target, count, trees):
    selector = CorrelationUtilitySelector(n_features=count, correlation_threshold=.90,
                                          random_state=seed, n_estimators=trees).fit(train, target)
    assert selector.n_features_out_ == count
    assert selector.maximum_selected_correlation_ <= .90 + 1e-12
    return selector.transform(train), selector.transform(valid), selector

def fit_predict(train_battles, valid_battles, target, amir_train, amir_valid, needed):
    inputs, selectors = {}, {}
    if needed.intersection(model_300):
        a, b, selectors['300'] = select_pair(amir_train, amir_valid, target, 300, 64)
        inputs['300'] = (a, b)
    if needed.intersection(aimilia_models) or '320 union logistic' in needed:
        dex = EmilyFeatures(str(chart), bundle='all').fit(train_battles)
        a, b = dex.transform(train_battles), dex.transform(valid_battles)
        assert a.shape[1] == b.shape[1] == 276
        inputs.update({key: (a.iloc[:, :size], b.iloc[:, :size]) for key, size in aimilia_sizes.items()})
    if '320 union logistic' in needed:
        union_train = pd.concat([a.add_prefix('emily__'), amir_train.add_prefix('amir__amir__')], axis=1)
        union_valid = pd.concat([b.add_prefix('emily__'), amir_valid.add_prefix('amir__amir__')], axis=1)
        a, b, selectors['320'] = select_pair(union_train, union_valid, target, 320, 32)
        inputs['320'] = (a, b)
    probabilities = {}
    for name in members:
        if name in needed:
            train, valid = inputs[name.split()[0]]
            model, params = members[name]
            pipeline = make_pipeline(model, seed=seed, params=params).fit(train, target)
            probabilities[name] = pipeline.predict_proba(valid)[:, 1]
    return probabilities, selectors

def combine(probabilities, recipe):
    return sum(weight * probabilities[name] for name, weight in recipe.items())

def decide(probability):
    return (probability > .50).astype(int)

### Refit and score each fold

Compute the deterministic per-battle Amir measurements once. Aimilia's species
lookup and both selectors use each fold's fitting rows. Store one out-of-fold
probability per development battle and candidate.

In [ ]:
if (run / 'holdout_metrics.json').exists():
    raise RuntimeError('This run already scored the holdout. Keep its development comparison frozen.')

amir_dev = amir_matrix(dev_battles)
assert amir_dev.shape == (len(dev), 753)
oof = {name: np.full(len(dev), np.nan) for name in members}
fold_features = {}
with threadpool_limits(limits=2):
    for fold, (fit_rows, valid_rows) in enumerate(folds, 1):
        probabilities, selectors = fit_predict(
            dev_battles.iloc[fit_rows], dev_battles.iloc[valid_rows], y_dev[fit_rows],
            amir_dev.iloc[fit_rows], amir_dev.iloc[valid_rows], set(members),
        )
        for name, probability in probabilities.items():
            oof[name][valid_rows] = probability
        fold_features[str(fold)] = {key: selector.selected_features_.tolist()
                                    for key, selector in selectors.items()}
        print(f'Fold {fold}/{n_folds} complete', flush=True)

oof.update({name: combine(oof, recipe) for name, recipe in recipes.items()})
assert all(np.isfinite(p).all() and ((0 <= p) & (p <= 1)).all() for p in oof.values())
fold_id = split.loc[dev, 'validation_fold'].to_numpy()
pd.DataFrame({'battle_id': ids[dev], 'fold': fold_id, 'actual': y_dev, **oof}).to_csv(
    run / 'oof_probabilities.csv', index=False
)
(run / 'fold_features.json').write_text(json.dumps(fold_features, indent=2) + '\n')

## Results

Mean fold accuracy ranks the candidates. The table also shows pooled accuracy
and probability metrics from the same out-of-fold rows. Fold SD describes variation
across these folds. For the paired comparison, both models predict the same battles.

In [ ]:
fold_accuracy = pd.DataFrame({
    name: [accuracy_score(y_dev[valid], decide(oof[name][valid])) for _, valid in folds]
    for name in candidates
}, index=pd.Index(range(1, n_folds + 1), name='Fold'))

def metrics(target, probability):
    prediction = decide(probability)
    return {'Pooled accuracy': accuracy_score(target, prediction),
            'Balanced accuracy': balanced_accuracy_score(target, prediction),
            'ROC AUC': roc_auc_score(target, probability), 'Log loss': log_loss(target, probability),
            'Brier': brier_score_loss(target, probability)}

results = pd.DataFrame({name: {'Mean fold accuracy': fold_accuracy[name].mean(),
                               'Fold SD': fold_accuracy[name].std(), **metrics(y_dev, oof[name])}
                        for name in candidates}).T
results.index.name = 'Candidate'
results = results.sort_values(['Mean fold accuracy', 'Log loss'], ascending=[False, True])
chosen = results.index[0]
reference = '300 compact vote' if chosen != '300 compact vote' else '106 baseline vote'
results.to_csv(run / 'cv_results.csv')
fold_accuracy.to_csv(run / 'fold_accuracy.csv')
formats = {
    'Mean fold accuracy': '{:.3%}', 'Fold SD': '{:.2%}', 'Pooled accuracy': '{:.3%}',
    'Balanced accuracy': '{:.3%}', 'ROC AUC': '{:.4f}', 'Log loss': '{:.4f}', 'Brier': '{:.4f}',
}
display(results.assign(**{column: results[column].map(pattern.format)
                          for column, pattern in formats.items()}))
chosen_correct = decide(oof[chosen]) == y_dev
reference_correct = decide(oof[reference]) == y_dev
corrected = int((chosen_correct & ~reference_correct).sum())
regressed = int((reference_correct & ~chosen_correct).sum())
display(Markdown(f'**{chosen} vs {reference}.** Corrected {corrected} errors and introduced '
                 f'{regressed} on {len(y_dev):,} paired battles ({corrected-regressed:+d} net).'))

In [ ]:
labels = results.index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), layout='constrained')
ax = axes[0]
for row, name in enumerate(labels):
    values = fold_accuracy[name].to_numpy()
    ax.scatter(values, np.full(n_folds, row), s=24, facecolors='none', edgecolors=blue)
    ax.scatter(values.mean(), row, s=35, marker='D', color=ink)
ax.set(yticks=range(len(labels)), yticklabels=labels, xlabel='Accuracy',
       title='Grouped development folds')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.grid(axis='x', color='#DDE5EB')
ax.text(0, -.16, 'Open circles = folds   ◆ = mean', transform=ax.transAxes, color=ink)

ax = axes[1]
ax.plot([0, 1], [0, 1], ':', color=ink, label='Perfect calibration')
for name, color, marker in [(chosen, blue, 'o'), (reference, gold, 's')]:
    observed, predicted = calibration_curve(y_dev, oof[name], n_bins=8, strategy='quantile')
    ax.plot(predicted, observed, marker=marker, color=color, label=name)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean predicted P1 win probability',
       ylabel='Observed P1 win rate', title='Out-of-fold calibration')
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.grid(color='#DDE5EB')
ax.legend(loc='upper left', fontsize=9)
plt.show()

### How sensitive is accuracy to the decision threshold?

The final o3 review suggested checking the 0.50 cutoff. The table applies five fixed
cutoffs to the selected model's saved out-of-fold probabilities. On this development
set, moving the cutoff by up to 0.04 changes pooled accuracy by less than half a
percentage point. Picking the best cutoff from these same outcomes would give an
optimistic estimate, so predictions continue to use 0.50.

In [ ]:
thresholds = (.46, .48, .50, .52, .54)
threshold_accuracy = pd.Series(
    {threshold: accuracy_score(y_dev, oof[chosen] > threshold) for threshold in thresholds},
    name='Pooled accuracy',
).rename_axis('P1 win threshold')
display(pd.DataFrame({
    'Pooled accuracy': threshold_accuracy.map('{:.2%}'.format),
    'Change from 0.50': (100 * (threshold_accuracy - threshold_accuracy.loc[.50])).map('{:+.2f} pp'.format),
}))

### Inspect the compact features

The development-fit dictionary is a readable reference. The fold-level audit
records which columns each validation fit actually used. Utility rank summarizes
the selector's heuristic.

In [ ]:
with threadpool_limits(limits=2):
    compact = CorrelationUtilitySelector(n_features=300, correlation_threshold=.90,
                                         random_state=seed).fit(amir_dev, y_dev)
definitions = feature_dictionary(compact.selected_features_)
frequency = pd.Series([name for fold in fold_features.values() for name in fold['300']]).value_counts()
audit = compact.audit_.merge(definitions, on='feature', how='left')
audit['folds selected'] = audit['feature'].map(frequency).fillna(0).astype(int)
audit.to_csv(run / 'compact_feature_audit.csv', index=False)
definitions.to_csv(run / 'compact_feature_dictionary.csv', index=False)
display(audit.loc[audit['selected'], ['feature', 'category', 'utility_rank', 'folds selected']]
        .sort_values('utility_rank').head(8).set_index('utility_rank'))
print(f'{(frequency == n_folds).sum()} compact features selected in all {n_folds} folds. '
      f'Development maximum selected |r| = {compact.maximum_selected_correlation_:.3f}.')

## Takeaways

The selected recipe improved 4 of 5 folds against 300 compact vote. It corrected 185 errors and introduced 153. Its out-of-fold ROC AUC was 0.752. The gains are exploratory because model choices reuse these development folds.

Earlier evidence is retained in the 300-feature run `20261105_003610_features300`
the 757-feature expansion run
`20261104_232631_018954_feature_expansion`, and the o3-reviewed run `20261105_122207_113551_emily_enhanced` under
`outputs/amirhosein/`, with original notebooks in the local archive under `outputs/amirhosein/archive/`.
The 757-feature expansion, sparse identity variant and prior numerical scores
remain in those experiment records. The table above uses fresh fits with the
legacy lookup excluded.

## Optional holdout and submission

The holdout switch scores the selected recipe after fitting development rows.
The submission switch refits on all official labeled battles, aligns binary
predictions to the sample's ID order, and writes one local CSV. Both switches
default to off. Keep the candidate fixed once a run has scored its holdout.

The run manifest records the execution switches and fingerprints the kernel inputs from setup through the audit cell. This also captures changes made in a runner’s copy.

In [ ]:
EVALUATE_HOLDOUT = False
MAKE_SUBMISSION = False

def predict_new(train_battles, valid_battles, target, recipe, amir_train=None):
    needed = set(recipe)
    needs_amir = bool(needed.intersection(model_300) or '320 union logistic' in needed)
    if needs_amir and amir_train is None:
        amir_train = amir_matrix(train_battles)
    amir_valid = amir_matrix(valid_battles) if needs_amir else None
    with threadpool_limits(limits=2):
        probabilities, selectors = fit_predict(
            train_battles, valid_battles, target, amir_train, amir_valid, needed,
        )
    return combine(probabilities, recipe), selectors

recipe = recipes.get(chosen, {chosen: 1.0})
holdout_path = run / 'holdout_metrics.json'
if EVALUATE_HOLDOUT:
    if holdout_path.exists():
        raise RuntimeError('This run has already scored the holdout.')
    p_holdout, _ = predict_new(dev_battles, battles.iloc[holdout], y_dev, recipe, amir_dev)
    record = {'candidate': chosen, 'battles': len(holdout), **metrics(y[holdout], p_holdout)}
    holdout_path.write_text(json.dumps(record, indent=2) + '\n')
    print(f"Holdout accuracy: {record['Pooled accuracy']:.2%}")

submission_path = run / 'submission.csv'
if MAKE_SUBMISSION:
    if holdout_path.exists() and json.loads(holdout_path.read_text())['candidate'] != chosen:
        raise RuntimeError('Keep the holdout-scored candidate fixed.')
    test = list(iter_battles(data / 'test.jsonl'))
    test_ids = pd.Index([battle['battle_id'] for battle in test])
    sample = pd.read_csv(data / 'sample_submission.csv')
    assert list(sample.columns) == ['battle_id', 'player_won']
    assert test_ids.is_unique and sample['battle_id'].is_unique
    assert set(test_ids) == set(sample['battle_id'])
    p_test, selectors = predict_new(battles, pd.DataFrame({'battle': test}), y, recipe)
    assert np.isfinite(p_test).all() and ((0 <= p_test) & (p_test <= 1)).all()
    sample['player_won'] = sample['battle_id'].map(pd.Series(decide(p_test), index=test_ids))
    assert sample['player_won'].isin([0, 1]).all()
    (run / 'submission_features.json').write_text(json.dumps({
        key: selector.selected_features_.tolist() for key, selector in selectors.items()
    }, indent=2) + '\n')
    sample.to_csv(submission_path, index=False)
    print(f'Submission written with {len(sample):,} rows.')

def sha256(path):
    with path.open('rb') as file:
        return file_digest(file, 'sha256').hexdigest()

source_paths = sorted([
    *support.glob('*.py'), *(root / 'src/pokemon_challenge').rglob('*.py'),
    *(root / 'notebooks/aimilia').glob('*.py'),
    root / 'notebooks/leonard/ablation_extensions.py',
    root / 'notebooks/leonard/progressive_features.py',
    root / 'configs/validation.json', root / 'requirements.txt',
])
inputs = [data / 'train.jsonl', chart]
if MAKE_SUBMISSION:
    inputs += [data / 'test.jsonl', data / 'sample_submission.csv']
executed_inputs = kernel.history_manager.input_hist_raw[audit_history_start:] if kernel else []
manifest = {
    'notebook_execution': {
        'kernel_inputs_sha256': hash_bytes(json.dumps(executed_inputs).encode()).hexdigest() if executed_inputs else None,
        'input_count': len(executed_inputs),
        'evaluate_holdout': EVALUATE_HOLDOUT,
        'make_submission': MAKE_SUBMISSION,
    },
    'candidate': chosen, 'weights': recipe, 'member_models': members, 'vote_recipes': recipes,
    'validation': {**settings, 'splitter': 'StratifiedGroupKFold'},
    'selector': {'max_abs_pearson': .90, 'trees_by_feature_count': {300: 64, 320: 32}},
    'holdout_scored': holdout_path.exists(), 'submission_written': submission_path.exists(),
    'python': sys.version.split()[0],
    'packages': {pkg: version(pkg) for pkg in ('numpy', 'pandas', 'scikit-learn', 'xgboost')},
    'input_sha256': {path.name: sha256(path) for path in inputs},
    'splits_sha256': sha256(run / 'splits.csv'),
    'source_sha256': {str(path.relative_to(root)): sha256(path) for path in source_paths},
}
(run / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(f'Audit saved under outputs/amirhosein/{run.name}/')